# Health, readiness and smoke checks

This notebook runs the Python blocks of the lesson *Run the whole stack with Docker Compose*. It does **not** run Docker. It starts a tiny web service on `127.0.0.1` with the standard library, so nothing has to be installed and no key is needed. The outputs saved in the cells come from a real run.

## 1. Alive is not the same as ready

In [ ]:
import http.client, json, threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

DOCS = {"returns-2": "Returns are accepted for 30 days.", "pay-1": "Refunds are paid within 5 working days."}

class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        index = self.server.index                   # None until it has been loaded
        if self.path == "/healthz":
            return self.reply(200, {"alive": True})
        if index is None:
            return self.reply(503, {"ready": False})
        if self.path == "/readyz":
            return self.reply(200, {"ready": True})
        words = set(self.path.removeprefix("/ask?q=").split("+"))
        best = max(index, key=lambda i: len(words & set(index[i].lower().strip(".").split())))
        self.reply(200, {"answer": index[best], "citation": best})

    def reply(self, status, body):
        data = json.dumps(body).encode()
        self.send_response(status)
        self.send_header("Content-Length", str(len(data)))
        self.end_headers()
        self.wfile.write(data)

    def log_message(self, *args):                   # keep the output clean
        pass

def start_server():
    server = ThreadingHTTPServer(("127.0.0.1", 0), Handler)      # port 0: any free port
    server.index = None
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server

def get(server, path):
    connection = http.client.HTTPConnection("127.0.0.1", server.server_address[1], timeout=5)
    connection.request("GET", path)
    response = connection.getresponse()
    result = response.status, json.loads(response.read())
    connection.close()
    return result

server = start_server()
for path in ("/healthz", "/readyz", "/ask?q=returns+days"):
    print("before loading", path.split("?")[0], get(server, path))
server.index = dict(DOCS)
print("after loading  /readyz", get(server, "/readyz"))
server.shutdown()
server.server_close()

before loading /healthz (200, {'alive': True})
before loading /readyz (503, {'ready': False})
before loading /ask (503, {'ready': False})
after loading  /readyz (200, {'ready': True})


## 2. Waiting a bounded number of attempts

In [ ]:
import http.client, json, threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

DOCS = {"returns-2": "Returns are accepted for 30 days.", "pay-1": "Refunds are paid within 5 working days."}

class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        index = self.server.index                   # None until it has been loaded
        if self.path == "/healthz":
            return self.reply(200, {"alive": True})
        if index is None:
            return self.reply(503, {"ready": False})
        if self.path == "/readyz":
            return self.reply(200, {"ready": True})
        words = set(self.path.removeprefix("/ask?q=").split("+"))
        best = max(index, key=lambda i: len(words & set(index[i].lower().strip(".").split())))
        self.reply(200, {"answer": index[best], "citation": best})

    def reply(self, status, body):
        data = json.dumps(body).encode()
        self.send_response(status)
        self.send_header("Content-Length", str(len(data)))
        self.end_headers()
        self.wfile.write(data)

    def log_message(self, *args):                   # keep the output clean
        pass

def start_server():
    server = ThreadingHTTPServer(("127.0.0.1", 0), Handler)      # port 0: any free port
    server.index = None
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server

def get(server, path):
    connection = http.client.HTTPConnection("127.0.0.1", server.server_address[1], timeout=5)
    connection.request("GET", path)
    response = connection.getresponse()
    result = response.status, json.loads(response.read())
    connection.close()
    return result

def wait_ready(server, attempts, between):
    for attempt in range(1, attempts + 1):
        if get(server, "/readyz")[0] == 200:
            return attempt
        between(attempt)                            # a real script would sleep here
    return None

for finishes_after in (3, None):
    server = start_server()
    def between(attempt):
        if attempt == finishes_after:
            server.index = dict(DOCS)
    answered = wait_ready(server, attempts=5, between=between)
    print("ready on attempt", answered) if answered else print("gave up after 5 attempts")
    server.shutdown()
    server.server_close()

ready on attempt 4
gave up after 5 attempts


## 3. A smoke test that checks the content

In [ ]:
import http.client, json, threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

DOCS = {"returns-2": "Returns are accepted for 30 days.", "pay-1": "Refunds are paid within 5 working days."}

class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        index = self.server.index                   # None until it has been loaded
        if self.path == "/healthz":
            return self.reply(200, {"alive": True})
        if index is None:
            return self.reply(503, {"ready": False})
        if self.path == "/readyz":
            return self.reply(200, {"ready": True})
        words = set(self.path.removeprefix("/ask?q=").split("+"))
        best = max(index, key=lambda i: len(words & set(index[i].lower().strip(".").split())))
        self.reply(200, {"answer": index[best], "citation": best})

    def reply(self, status, body):
        data = json.dumps(body).encode()
        self.send_response(status)
        self.send_header("Content-Length", str(len(data)))
        self.end_headers()
        self.wfile.write(data)

    def log_message(self, *args):                   # keep the output clean
        pass

def start_server():
    server = ThreadingHTTPServer(("127.0.0.1", 0), Handler)      # port 0: any free port
    server.index = None
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server

def get(server, path):
    connection = http.client.HTTPConnection("127.0.0.1", server.server_address[1], timeout=5)
    connection.request("GET", path)
    response = connection.getresponse()
    result = response.status, json.loads(response.read())
    connection.close()
    return result

def wait_ready(server, attempts, between):
    for attempt in range(1, attempts + 1):
        if get(server, "/readyz")[0] == 200:
            return attempt
        between(attempt)                            # a real script would sleep here
    return None

for name, docs in (("good index", DOCS), ("old index", {"returns-1": "Returns are accepted for 14 days."})):
    server = start_server()
    server.index = dict(docs)
    try:
        assert wait_ready(server, 3, lambda attempt: None), "never ready"
        status, body = get(server, "/ask?q=returns+accepted+days")
        passed = status == 200 and "30 days" in body["answer"] and body["citation"] == "returns-2"
        print(name, "->", "PASS" if passed else "FAIL", body["citation"], "|", body["answer"])
    finally:
        server.shutdown()
        server.server_close()

good index -> PASS returns-2 | Returns are accepted for 30 days.
old index -> FAIL returns-1 | Returns are accepted for 14 days.


## Try next

Add a second question with its expected citation to the smoke test. Use invented data only.